# OHLCV Time Series Classification for Price Movement Prediction

## 🎯 Objective
This notebook develops a research-grade ML pipeline that:
- Takes OHLCV (Open, High, Low, Close, Volume) time series data as input
- Engineers features from technical indicators and statistical transformations
- Labels each timestamp according to future price movement over N bars
- Trains and evaluates multiple ML models to predict 3-class directional movement
- Includes extensive visualizations, intermediate result tables, and model diagnostics

---

# Preparation

## Cell 1: Imports and Global Config

In [ ]:
# Cell 1: Imports and Global Config

# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
import xgboost as xgb
import lightgbm as lgb

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

# Technical Analysis
import ta

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Hyperparameter Optimization
import optuna

# Model Interpretability
import shap

# Utilities
from tqdm import tqdm
import joblib
from datetime import datetime, timedelta

# Global Constants
N = 10           # prediction horizon (bars)
P_PCT = 1.0      # threshold in percent (e.g., 1%)
RANDOM_SEED = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2   # of remaining after test split

# Set random seeds for reproducibility
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Matplotlib style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print(f"Configuration loaded successfully!")
print(f"Prediction horizon: {N} bars")
print(f"Price threshold: {P_PCT}%")
print(f"Random seed: {RANDOM_SEED}")
print(f"TensorFlow version: {tf.__version__}")

## Cell 2: Load and Inspect Raw OHLCV Data

In [ ]:
# Cell 2: Load and Inspect Raw OHLCV Data

def generate_sample_ohlcv_data(n_days=1000, start_price=100, volatility=0.02):
    """
    Generate synthetic OHLCV data for demonstration purposes.
    In practice, you would load real data from CSV/Parquet files.
    """
    dates = pd.date_range(start='2020-01-01', periods=n_days, freq='D')
    
    # Generate price series with trend and volatility
    np.random.seed(RANDOM_SEED)
    returns = np.random.normal(0.0005, volatility, n_days)  # Small positive drift
    
    # Add some autocorrelation to make it more realistic
    for i in range(1, len(returns)):
        returns[i] += 0.1 * returns[i-1]
    
    close_prices = [start_price]
    for ret in returns[1:]:
        close_prices.append(close_prices[-1] * (1 + ret))
    
    # Generate OHLCV data
    data = []
    for i, (date, close) in enumerate(zip(dates, close_prices)):
        # Generate realistic OHLC from close price
        daily_range = abs(np.random.normal(0, close * 0.01))
        high = close + np.random.uniform(0, daily_range)
        low = close - np.random.uniform(0, daily_range)
        open_price = low + np.random.uniform(0, high - low)
        
        # Ensure OHLC relationships are maintained
        high = max(high, open_price, close)
        low = min(low, open_price, close)
        
        # Volume correlated with price volatility
        volume = int(np.random.lognormal(10, 1) * (1 + abs(returns[i]) * 10))
        
        data.append({
            'date': date,
            'open': open_price,
            'high': high,
            'low': low,
            'close': close,
            'volume': volume
        })
    
    return pd.DataFrame(data)

# Load data (replace with your actual data loading)
print("Loading OHLCV data...")
df = generate_sample_ohlcv_data(n_days=1000)
df.set_index('date', inplace=True)

# Display basic information
print("\\n" + "="*50)
print("DATA INSPECTION")
print("="*50)

print("\\nDataFrame Info:")
print(df.info())

print("\\nDataFrame Description:")
print(df.describe())

print("\\nFirst 5 rows:")
print(df.head())

print("\\nLast 5 rows:")
print(df.tail())

# Plot price and volume over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

# Price plot
ax1.plot(df.index, df['close'], label='Close Price', linewidth=1.5, color='blue')
ax1.set_title('Close Price Over Time', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price ($)', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Volume plot
ax2.bar(df.index, df['volume'], alpha=0.7, color='orange', width=1)
ax2.set_title('Volume Over Time', fontsize=14, fontweight='bold')
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Volume', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\\nData shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Missing values: {df.isnull().sum().sum()}")

## Cell 3: Label Generation (Target Definition)

In [ ]:
# Cell 3: Label Generation (Target Definition)

print("="*50)
print("LABEL GENERATION")
print("="*50)

# Compute future return over N periods
df['future_ret'] = df['close'].pct_change(periods=N).shift(-N)

def label_movement(ret, p=P_PCT/100):
    """
    Label price movements based on future returns.
    
    Args:
        ret: Future return
        p: Threshold percentage (as decimal)
    
    Returns:
        1: Up movement (>= p%)
        -1: Down movement (<= -p%)
        0: Flat movement (between -p% and p%)
    """
    if pd.isna(ret):
        return np.nan
    elif ret >= p:
        return 1      # up
    elif ret <= -p:
        return -1     # down
    else:
        return 0      # flat

# Apply labeling function
df['label'] = df['future_ret'].apply(lambda x: label_movement(x, P_PCT/100))

# Drop last N rows (no future label available)
df_labeled = df.dropna(subset=['label']).copy()
df_labeled['label'] = df_labeled['label'].astype(int)

print(f"Original data shape: {df.shape}")
print(f"Labeled data shape: {df_labeled.shape}")
print(f"Rows dropped due to future horizon: {df.shape[0] - df_labeled.shape[0]}")

# Show label distribution
label_counts = df_labeled['label'].value_counts().sort_index()
label_props = df_labeled['label'].value_counts(normalize=True).sort_index()

print("\\nLabel Distribution:")
print("-" * 30)
label_mapping = {-1: 'Down', 0: 'Flat', 1: 'Up'}
for label in [-1, 0, 1]:
    count = label_counts.get(label, 0)
    prop = label_props.get(label, 0)
    print(f"{label_mapping[label]:>4} ({label:2d}): {count:4d} samples ({prop:6.1%})")

# Visualize label distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Bar plot of counts
labels = [label_mapping[i] for i in [-1, 0, 1]]
counts = [label_counts.get(i, 0) for i in [-1, 0, 1]]
colors = ['red', 'gray', 'green']

bars = ax1.bar(labels, counts, color=colors, alpha=0.7)
ax1.set_title('Label Distribution (Counts)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Samples', fontsize=12)
ax1.grid(True, alpha=0.3)

# Add count labels on bars
for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
             f'{count}', ha='center', va='bottom', fontweight='bold')

# Pie chart of proportions
props = [label_props.get(i, 0) for i in [-1, 0, 1]]
ax2.pie(props, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax2.set_title('Label Distribution (Proportions)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Show some examples of labeled data
print("\\nSample of labeled data:")
print(df_labeled[['close', 'future_ret', 'label']].head(10))

# Update main dataframe
df = df_labeled.copy()
print(f"\\nFinal dataset shape: {df.shape}")

## Cell 4: Feature Engineering

In [ ]:
# Cell 4: Feature Engineering

print("="*50)
print("FEATURE ENGINEERING")
print("="*50)

# Store original columns
original_cols = df.columns.tolist()
print(f"Starting with {len(original_cols)} columns: {original_cols}")

# 1. Raw transformations
print("\\n1. Raw transformations...")
df['log_return'] = np.log(df['close'] / df['close'].shift(1))
df['hl_ratio'] = (df['high'] - df['low']) / df['close']
df['oc_ratio'] = (df['close'] - df['open']) / df['open']
df['candle_body'] = abs(df['close'] - df['open']) / df['close']
df['upper_shadow'] = (df['high'] - np.maximum(df['open'], df['close'])) / df['close']
df['lower_shadow'] = (np.minimum(df['open'], df['close']) - df['low']) / df['close']
df['price_position'] = (df['close'] - df['low']) / (df['high'] - df['low'])

# 2. Rolling statistics
print("2. Rolling statistics...")
windows = [5, 10, 20]
for window in windows:
    # Price-based rolling features
    df[f'close_sma_{window}'] = df['close'].rolling(window).mean()
    df[f'close_std_{window}'] = df['close'].rolling(window).std()
    df[f'return_sma_{window}'] = df['log_return'].rolling(window).mean()
    df[f'return_std_{window}'] = df['log_return'].rolling(window).std()
    df[f'volume_sma_{window}'] = df['volume'].rolling(window).mean()
    
    # Price relative to moving average
    df[f'close_vs_sma_{window}'] = df['close'] / df[f'close_sma_{window}'] - 1
    
    # Rolling min/max
    df[f'close_min_{window}'] = df['close'].rolling(window).min()
    df[f'close_max_{window}'] = df['close'].rolling(window).max()
    df[f'close_vs_min_{window}'] = (df['close'] - df[f'close_min_{window}']) / df[f'close_min_{window}']
    df[f'close_vs_max_{window}'] = (df['close'] - df[f'close_max_{window}']) / df[f'close_max_{window}']

# 3. Technical indicators using TA library
print("3. Technical indicators...")
try:
    # Add all technical indicators
    df = ta.add_all_ta_features(df, open="open", high="high", low="low", 
                               close="close", volume="volume", fillna=True)
    print("   Added all TA indicators successfully")
except Exception as e:
    print(f"   Error adding TA indicators: {e}")
    # Add key indicators manually
    df['rsi'] = ta.momentum.RSIIndicator(df['close'], window=14).rsi()
    df['macd'] = ta.trend.MACD(df['close']).macd()
    df['macd_signal'] = ta.trend.MACD(df['close']).macd_signal()
    df['bb_high'] = ta.volatility.BollingerBands(df['close']).bollinger_hband()
    df['bb_low'] = ta.volatility.BollingerBands(df['close']).bollinger_lband()
    df['bb_mid'] = ta.volatility.BollingerBands(df['close']).bollinger_mavg()
    df['atr'] = ta.volatility.AverageTrueRange(df['high'], df['low'], df['close']).average_true_range()
    df['ema_10'] = ta.trend.EMAIndicator(df['close'], window=10).ema_indicator()
    df['ema_30'] = ta.trend.EMAIndicator(df['close'], window=30).ema_indicator()
    df['stoch_k'] = ta.momentum.StochasticOscillator(df['high'], df['low'], df['close']).stoch()
    df['stoch_d'] = ta.momentum.StochasticOscillator(df['high'], df['low'], df['close']).stoch_signal()
    
    # Bollinger Band position
    df['bb_position'] = (df['close'] - df['bb_low']) / (df['bb_high'] - df['bb_low'])

# 4. Lag features
print("4. Lag features...")
lag_features = ['close', 'log_return', 'volume', 'hl_ratio']
lag_periods = [1, 2, 3, 5]

for feature in lag_features:
    if feature in df.columns:
        for lag in lag_periods:
            df[f'{feature}_lag_{lag}'] = df[feature].shift(lag)

# 5. Time-based features (if datetime index)
print("5. Time-based features...")
if isinstance(df.index, pd.DatetimeIndex):
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    df['day_of_month'] = df.index.day
    
    # Cyclical encoding
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Drop rows with NaN values
print("\\nCleaning data...")
print(f"Shape before cleaning: {df.shape}")
df_clean = df.dropna()
print(f"Shape after cleaning: {df_clean.shape}")
print(f"Rows dropped: {df.shape[0] - df_clean.shape[0]}")

# Get feature columns (exclude target and original OHLCV)
exclude_cols = ['open', 'high', 'low', 'close', 'volume', 'future_ret', 'label']
feature_cols = [col for col in df_clean.columns if col not in exclude_cols]

print(f"\\nTotal features created: {len(feature_cols)}")
print(f"Feature categories:")
print(f"  - Raw transformations: {len([c for c in feature_cols if any(x in c for x in ['log_return', 'ratio', 'body', 'shadow', 'position'])])}")
print(f"  - Rolling statistics: {len([c for c in feature_cols if any(x in c for x in ['sma', 'std', 'min', 'max', 'vs'])])}")
print(f"  - Technical indicators: {len([c for c in feature_cols if any(x in c for x in ['rsi', 'macd', 'bb_', 'atr', 'ema', 'stoch', 'trend', 'momentum', 'volatility'])])}")
print(f"  - Lag features: {len([c for c in feature_cols if 'lag' in c])}")
print(f"  - Time features: {len([c for c in feature_cols if any(x in c for x in ['day', 'month', 'sin', 'cos'])])}")

# Display sample of feature matrix
print("\\nSample of feature matrix:")
print(df_clean[feature_cols].head())

# Update main dataframe
df = df_clean.copy()
print(f"\\nFinal dataset shape: {df.shape}")
print(f"Features available: {len(feature_cols)}")

## Cell 5: Exploratory Data Analysis (EDA)

In [ ]:
# Cell 5: Exploratory Data Analysis (EDA)

print("="*50)
print("EXPLORATORY DATA ANALYSIS")
print("="*50)

# Select top predictive features using mutual information
print("Selecting top predictive features...")
X_temp = df[feature_cols].fillna(0)
y_temp = df['label']

# Calculate mutual information scores
mi_scores = mutual_info_classif(X_temp, y_temp, random_state=RANDOM_SEED)
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

top_features = feature_importance.head(20)['feature'].tolist()
print(f"Top 20 most predictive features:")
for i, (_, row) in enumerate(feature_importance.head(20).iterrows(), 1):
    print(f"{i:2d}. {row['feature']:<25} (MI: {row['mi_score']:.4f})")

# 1. Correlation heatmap of top features
print("\n1. Creating correlation heatmap...")
fig, ax = plt.subplots(figsize=(15, 12))
correlation_matrix = df[top_features].corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": .8}, fmt='.2f')
ax.set_title('Correlation Heatmap of Top 20 Predictive Features', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# 2. Sample features vs price plot
print("\n2. Feature vs Price Analysis...")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

# Select some interesting features to plot
plot_features = []
for feature_type in ['rsi', 'macd', 'bb_position', 'log_return']:
    matching_features = [f for f in top_features if feature_type in f.lower()]
    if matching_features:
        plot_features.append(matching_features[0])

if len(plot_features) < 4:
    plot_features = top_features[:4]

for i, feature in enumerate(plot_features[:4]):
    ax1 = axes[i]
    ax2 = ax1.twinx()
    
    # Plot feature
    ax1.plot(df.index[-200:], df[feature].iloc[-200:], color='blue', alpha=0.7, label=feature)
    ax1.set_ylabel(feature, color='blue', fontsize=10)
    ax1.tick_params(axis='y', labelcolor='blue')
    
    # Plot price
    ax2.plot(df.index[-200:], df['close'].iloc[-200:], color='red', alpha=0.7, label='Close Price')
    ax2.set_ylabel('Close Price', color='red', fontsize=10)
    ax2.tick_params(axis='y', labelcolor='red')
    
    ax1.set_title(f'{feature} vs Close Price (Last 200 periods)', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 3. Feature distributions by label class
print("\n3. Feature distributions by label class...")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

label_mapping = {-1: 'Down', 0: 'Flat', 1: 'Up'}
colors = ['red', 'gray', 'green']

for i, feature in enumerate(top_features[:4]):
    ax = axes[i]
    
    for label_val, color in zip([-1, 0, 1], colors):
        data = df[df['label'] == label_val][feature].dropna()
        ax.hist(data, bins=30, alpha=0.6, label=f'{label_mapping[label_val]} (n={len(data)})',
                color=color, density=True)
    
    ax.set_title(f'Distribution of {feature} by Label', fontsize=12, fontweight='bold')
    ax.set_xlabel(feature, fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. Feature statistics by class
print("\n4. Feature statistics by class:")
print("-" * 60)
stats_by_class = df.groupby('label')[top_features[:5]].agg(['mean', 'std']).round(4)
print(stats_by_class)

# 5. Class balance over time
print("\n5. Class balance over time...")
if isinstance(df.index, pd.DatetimeIndex):
    # Create monthly resampling with proper handling
    df_monthly = df.resample('M').apply(lambda x: pd.Series({
        'down_count': (x['label'] == -1).sum(),
        'flat_count': (x['label'] == 0).sum(), 
        'up_count': (x['label'] == 1).sum(),
        'total_count': len(x)
    }))
    
    # Calculate proportions
    df_monthly['down_prop'] = df_monthly['down_count'] / df_monthly['total_count']
    df_monthly['flat_prop'] = df_monthly['flat_count'] / df_monthly['total_count']
    df_monthly['up_prop'] = df_monthly['up_count'] / df_monthly['total_count']
    
    # Fill NaN values with 0
    df_monthly = df_monthly.fillna(0)
    
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.stackplot(df_monthly.index, 
                df_monthly['down_prop'], 
                df_monthly['flat_prop'], 
                df_monthly['up_prop'],
                labels=['Down', 'Flat', 'Up'],
                colors=['red', 'gray', 'green'], 
                alpha=0.7)
    
    ax.set_title('Class Distribution Over Time (Monthly)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Proportion', fontsize=12)
    ax.set_xlabel('Date', fontsize=12)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

print("\n" + "="*50)
print("EDA INSIGHTS:")
print("="*50)
print(f"• Dataset contains {len(feature_cols)} engineered features")
print(f"• Top predictive feature: {feature_importance.iloc[0]['feature']} (MI: {feature_importance.iloc[0]['mi_score']:.4f})")
print(f"• Feature correlations range from {correlation_matrix.min().min():.3f} to {correlation_matrix.max().max():.3f}")
print(f"• Class distribution: Down={df['label'].value_counts()[-1]}, Flat={df['label'].value_counts()[0]}, Up={df['label'].value_counts()[1]}")
print("• Review the plots above for feature behavior patterns by class")

## Cell 6: Train/Validation/Test Split (Temporal)

In [ ]:
# Cell 6: Train/Validation/Test Split (Temporal)

print("="*50)
print("TEMPORAL DATA SPLITTING")
print("="*50)

# Prepare features and target
X = df[feature_cols].copy()
y = df['label'].copy()

print(f"Total samples: {len(X)}")
print(f"Total features: {len(feature_cols)}")

# Calculate split indices (no shuffling - temporal split)
n_total = len(X)
n_test = int(n_total * TEST_SIZE)
n_val = int((n_total - n_test) * VAL_SIZE)
n_train = n_total - n_test - n_val

# Split indices
train_end = n_train
val_end = train_end + n_val

print(f"\\nSplit sizes:")
print(f"  Train: {n_train} samples ({n_train/n_total:.1%})")
print(f"  Validation: {n_val} samples ({n_val/n_total:.1%})")
print(f"  Test: {n_test} samples ({n_test/n_total:.1%})")

# Perform temporal split
X_train = X.iloc[:train_end].copy()
X_val = X.iloc[train_end:val_end].copy()
X_test = X.iloc[val_end:].copy()

y_train = y.iloc[:train_end].copy()
y_val = y.iloc[train_end:val_end].copy()
y_test = y.iloc[val_end:].copy()

# Verify splits
print(f"\\nVerifying splits:")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_val shape: {X_val.shape}")
print(f"  X_test shape: {X_test.shape}")
print(f"  Total: {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]} (should equal {n_total})")

# Show date ranges for each split (if datetime index)
if isinstance(df.index, pd.DatetimeIndex):
    print(f"\\nTemporal ranges:")
    print(f"  Train: {df.index[0]} to {df.index[train_end-1]}")
    print(f"  Val:   {df.index[train_end]} to {df.index[val_end-1]}")
    print(f"  Test:  {df.index[val_end]} to {df.index[-1]}")

# Check class distribution in each split
def print_class_distribution(y_split, split_name):
    counts = y_split.value_counts().sort_index()
    props = y_split.value_counts(normalize=True).sort_index()
    
    print(f"\\n{split_name} class distribution:")
    label_mapping = {-1: 'Down', 0: 'Flat', 1: 'Up'}
    for label in [-1, 0, 1]:
        count = counts.get(label, 0)
        prop = props.get(label, 0)
        print(f"  {label_mapping[label]:>4} ({label:2d}): {count:4d} samples ({prop:6.1%})")

print_class_distribution(y_train, "Train")
print_class_distribution(y_val, "Validation")
print_class_distribution(y_test, "Test")

# Create summary table
split_summary = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Size': [len(y_train), len(y_val), len(y_test)],
    'Percentage': [len(y_train)/n_total*100, len(y_val)/n_total*100, len(y_test)/n_total*100],
    'Down_Count': [y_train.value_counts().get(-1, 0), y_val.value_counts().get(-1, 0), y_test.value_counts().get(-1, 0)],
    'Flat_Count': [y_train.value_counts().get(0, 0), y_val.value_counts().get(0, 0), y_test.value_counts().get(0, 0)],
    'Up_Count': [y_train.value_counts().get(1, 0), y_val.value_counts().get(1, 0), y_test.value_counts().get(1, 0)]
})

split_summary['Down_Pct'] = split_summary['Down_Count'] / split_summary['Size'] * 100
split_summary['Flat_Pct'] = split_summary['Flat_Count'] / split_summary['Size'] * 100
split_summary['Up_Pct'] = split_summary['Up_Count'] / split_summary['Size'] * 100

print(f"\\nSplit Summary Table:")
print("=" * 80)
print(split_summary.round(1))

# Visualize split distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Split sizes
splits = ['Train', 'Validation', 'Test']
sizes = [len(y_train), len(y_val), len(y_test)]
colors_split = ['lightblue', 'lightgreen', 'lightcoral']

ax1.bar(splits, sizes, color=colors_split, alpha=0.7)
ax1.set_title('Dataset Split Sizes', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Samples', fontsize=12)
ax1.grid(True, alpha=0.3)

# Add count labels on bars
for i, (split, size) in enumerate(zip(splits, sizes)):
    ax1.text(i, size + size*0.01, f'{size}\\n({size/n_total:.1%})', 
             ha='center', va='bottom', fontweight='bold')

# Class distribution by split
x = np.arange(len(splits))
width = 0.25

down_counts = [y_train.value_counts().get(-1, 0), y_val.value_counts().get(-1, 0), y_test.value_counts().get(-1, 0)]
flat_counts = [y_train.value_counts().get(0, 0), y_val.value_counts().get(0, 0), y_test.value_counts().get(0, 0)]
up_counts = [y_train.value_counts().get(1, 0), y_val.value_counts().get(1, 0), y_test.value_counts().get(1, 0)]

ax2.bar(x - width, down_counts, width, label='Down', color='red', alpha=0.7)
ax2.bar(x, flat_counts, width, label='Flat', color='gray', alpha=0.7)
ax2.bar(x + width, up_counts, width, label='Up', color='green', alpha=0.7)

ax2.set_title('Class Distribution by Split', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Samples', fontsize=12)
ax2.set_xlabel('Split', fontsize=12)
ax2.set_xticks(x)
ax2.set_xticklabels(splits)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\\n✅ Data successfully split into train/validation/test sets")
print(f"   Ready for feature scaling and model training!")

## Cell 7: Feature Scaling

In [ ]:
# Cell 7: Feature Scaling

print("="*50)
print("FEATURE SCALING")
print("="*50)

# Initialize StandardScaler
scaler = StandardScaler()

print("Fitting scaler on training data only...")
print(f"Training set shape: {X_train.shape}")

# Fit scaler only on training data
scaler.fit(X_train)

# Transform all splits
print("Transforming all splits...")
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames to preserve feature names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=feature_cols, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

print(f"✅ Scaling completed successfully!")
print(f"   Scaled training set shape: {X_train_scaled.shape}")
print(f"   Scaled validation set shape: {X_val_scaled.shape}")
print(f"   Scaled test set shape: {X_test_scaled.shape}")

# Show scaling statistics
print(f"\\nScaling Statistics:")
print(f"  Features scaled: {len(feature_cols)}")
print(f"  Scaler mean shape: {scaler.mean_.shape}")
print(f"  Scaler scale shape: {scaler.scale_.shape}")

# Display sample of scaled data
print(f"\\nSample of original vs scaled data:")
print("Original (first 3 features, first 5 rows):")
print(X_train[feature_cols[:3]].head())

print("\\nScaled (first 3 features, first 5 rows):")
print(X_train_scaled[feature_cols[:3]].head())

# Verify scaling worked correctly
print(f"\\nScaling verification:")
print(f"  Training set mean (should be ~0): {X_train_scaled.mean().abs().max():.6f}")
print(f"  Training set std (should be ~1): {abs(X_train_scaled.std().mean() - 1):.6f}")

# Show distribution before and after scaling for a sample feature
sample_feature = feature_cols[0]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Before scaling
ax1.hist(X_train[sample_feature], bins=50, alpha=0.7, color='blue', density=True)
ax1.set_title(f'Before Scaling: {sample_feature}', fontsize=12, fontweight='bold')
ax1.set_xlabel('Value', fontsize=10)
ax1.set_ylabel('Density', fontsize=10)
ax1.grid(True, alpha=0.3)

# After scaling
ax2.hist(X_train_scaled[sample_feature], bins=50, alpha=0.7, color='green', density=True)
ax2.set_title(f'After Scaling: {sample_feature}', fontsize=12, fontweight='bold')
ax2.set_xlabel('Standardized Value', fontsize=10)
ax2.set_ylabel('Density', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Store feature names for later use
feature_names = feature_cols.copy()
print(f"\\nFeature names stored: {len(feature_names)} features")

# Summary statistics comparison
print(f"\\nSummary Statistics Comparison:")
print("=" * 60)
comparison_stats = pd.DataFrame({
    'Original_Mean': X_train[feature_cols[:5]].mean(),
    'Original_Std': X_train[feature_cols[:5]].std(),
    'Scaled_Mean': X_train_scaled[feature_cols[:5]].mean(),
    'Scaled_Std': X_train_scaled[feature_cols[:5]].std()
}).round(4)
print(comparison_stats)

print(f"\\n✅ Feature scaling completed and verified!")
print(f"   All features now have mean ≈ 0 and std ≈ 1 on training set")
print(f"   Ready for model training!")

## Cell 8: Baseline Models

In [ ]:
# Cell 8: Baseline Models

print("="*50)
print("BASELINE MODELS")
print("="*50)

# Initialize results dictionary
results = {}

def evaluate_model(model, X_train, X_val, X_test, y_train, y_val, y_test, model_name):
    """
    Train and evaluate a model, returning metrics for validation and test sets.
    """
    print(f"\\nTraining {model_name}...")
    
    # Train the model
    start_time = datetime.now()
    model.fit(X_train, y_train)
    train_time = (datetime.now() - start_time).total_seconds()
    
    # Make predictions
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics
    val_metrics = {
        'accuracy': accuracy_score(y_val, y_val_pred),
        'balanced_accuracy': balanced_accuracy_score(y_val, y_val_pred),
        'f1_macro': f1_score(y_val, y_val_pred, average='macro')
    }
    
    test_metrics = {
        'accuracy': accuracy_score(y_test, y_test_pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_test_pred),
        'f1_macro': f1_score(y_test, y_test_pred, average='macro')
    }
    
    print(f"  Training time: {train_time:.2f} seconds")
    print(f"  Validation Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"  Validation F1-Macro: {val_metrics['f1_macro']:.4f}")
    
    return {
        'model': model,
        'train_time': train_time,
        'val_metrics': val_metrics,
        'test_metrics': test_metrics,
        'val_predictions': y_val_pred,
        'test_predictions': y_test_pred
    }

# 1. Logistic Regression with class weights
print("\\n1. Logistic Regression")
print("-" * 30)
lr_model = LogisticRegression(
    random_state=RANDOM_SEED,
    class_weight='balanced',  # Handle class imbalance
    max_iter=1000,
    solver='lbfgs'
)

results['Logistic Regression'] = evaluate_model(
    lr_model, X_train_scaled, X_val_scaled, X_test_scaled, 
    y_train, y_val, y_test, 'Logistic Regression'
)

# 2. Random Forest
print("\\n2. Random Forest")
print("-" * 30)
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=RANDOM_SEED,
    class_weight='balanced',
    n_jobs=-1
)

results['Random Forest'] = evaluate_model(
    rf_model, X_train_scaled, X_val_scaled, X_test_scaled, 
    y_train, y_val, y_test, 'Random Forest'
)

# 3. XGBoost
print("\\n3. XGBoost")
print("-" * 30)

# Calculate class weights for XGBoost
class_counts = y_train.value_counts().sort_index()
total_samples = len(y_train)
class_weights = {cls: total_samples / (len(class_counts) * count) 
                for cls, count in class_counts.items()}

# Convert labels to 0, 1, 2 for XGBoost
y_train_xgb = y_train.map({-1: 0, 0: 1, 1: 2})
y_val_xgb = y_val.map({-1: 0, 0: 1, 1: 2})
y_test_xgb = y_test.map({-1: 0, 0: 1, 1: 2})

xgb_model = xgb.XGBClassifier(
    random_state=RANDOM_SEED,
    eval_metric='mlogloss',
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1
)

print(f"  Training XGBoost...")
start_time = datetime.now()
xgb_model.fit(X_train_scaled, y_train_xgb)
train_time = (datetime.now() - start_time).total_seconds()

# Make predictions and convert back to original labels
y_val_pred_xgb = xgb_model.predict(X_val_scaled)
y_test_pred_xgb = xgb_model.predict(X_test_scaled)

# Convert predictions back to original label format
label_map_back = {0: -1, 1: 0, 2: 1}
y_val_pred_orig = pd.Series(y_val_pred_xgb).map(label_map_back)
y_test_pred_orig = pd.Series(y_test_pred_xgb).map(label_map_back)

# Calculate metrics
val_metrics_xgb = {
    'accuracy': accuracy_score(y_val, y_val_pred_orig),
    'balanced_accuracy': balanced_accuracy_score(y_val, y_val_pred_orig),
    'f1_macro': f1_score(y_val, y_val_pred_orig, average='macro')
}

test_metrics_xgb = {
    'accuracy': accuracy_score(y_test, y_test_pred_orig),
    'balanced_accuracy': balanced_accuracy_score(y_test, y_test_pred_orig),
    'f1_macro': f1_score(y_test, y_test_pred_orig, average='macro')
}

print(f"  Training time: {train_time:.2f} seconds")
print(f"  Validation Accuracy: {val_metrics_xgb['accuracy']:.4f}")
print(f"  Validation F1-Macro: {val_metrics_xgb['f1_macro']:.4f}")

results['XGBoost'] = {
    'model': xgb_model,
    'train_time': train_time,
    'val_metrics': val_metrics_xgb,
    'test_metrics': test_metrics_xgb,
    'val_predictions': y_val_pred_orig,
    'test_predictions': y_test_pred_orig
}

# Create comparison table
print(f"\\n" + "="*50)
print("BASELINE MODEL COMPARISON")
print("="*50)

comparison_data = []
for model_name, result in results.items():
    comparison_data.append({
        'Model': model_name,
        'Train_Time_s': result['train_time'],
        'Val_Accuracy': result['val_metrics']['accuracy'],
        'Val_Balanced_Acc': result['val_metrics']['balanced_accuracy'],
        'Val_F1_Macro': result['val_metrics']['f1_macro'],
        'Test_Accuracy': result['test_metrics']['accuracy'],
        'Test_Balanced_Acc': result['test_metrics']['balanced_accuracy'],
        'Test_F1_Macro': result['test_metrics']['f1_macro']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.round(4)
print(comparison_df)

# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = ['Val_Accuracy', 'Val_Balanced_Acc', 'Val_F1_Macro']
metric_names = ['Accuracy', 'Balanced Accuracy', 'F1-Macro']

for i, (metric, name) in enumerate(zip(metrics, metric_names)):
    ax = axes[i]
    bars = ax.bar(comparison_df['Model'], comparison_df[metric], 
                  color=['lightblue', 'lightgreen', 'lightcoral'], alpha=0.7)
    ax.set_title(f'Validation {name}', fontsize=12, fontweight='bold')
    ax.set_ylabel(name, fontsize=10)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, comparison_df[metric]):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Rotate x-axis labels if needed
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

# Find best model based on validation F1-macro
best_model_name = comparison_df.loc[comparison_df['Val_F1_Macro'].idxmax(), 'Model']
best_score = comparison_df['Val_F1_Macro'].max()

print(f"\\n🏆 Best baseline model: {best_model_name}")
print(f"   Validation F1-Macro: {best_score:.4f}")
print(f"\\n✅ Baseline models trained and evaluated!")

## Cell 9: Neural Network Prototype (MLP)

In [ ]:
# Cell 9: Neural Network Prototype (MLP)

print("="*50)
print("NEURAL NETWORK (MLP)")
print("="*50)

# Convert labels to categorical for neural network
print("Preparing data for neural network...")

# Map labels to 0, 1, 2 for categorical encoding
label_mapping = {-1: 0, 0: 1, 1: 2}
y_train_nn = y_train.map(label_mapping)
y_val_nn = y_val.map(label_mapping)
y_test_nn = y_test.map(label_mapping)

# Convert to categorical
y_train_cat = to_categorical(y_train_nn, num_classes=3)
y_val_cat = to_categorical(y_val_nn, num_classes=3)
y_test_cat = to_categorical(y_test_nn, num_classes=3)

print(f"  Original labels shape: {y_train.shape}")
print(f"  Categorical labels shape: {y_train_cat.shape}")
print(f"  Number of features: {X_train_scaled.shape[1]}")

# Build the MLP model
print("\\nBuilding MLP architecture...")
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
print("\\nModel Architecture:")
model.summary()

# Set up early stopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Train the model
print("\\nTraining neural network...")
start_time = datetime.now()

history = model.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

train_time = (datetime.now() - start_time).total_seconds()
print(f"\\nTraining completed in {train_time:.2f} seconds")

# Make predictions
print("\\nMaking predictions...")
y_val_pred_proba = model.predict(X_val_scaled, verbose=0)
y_test_pred_proba = model.predict(X_test_scaled, verbose=0)

# Convert probabilities to class predictions
y_val_pred_nn = np.argmax(y_val_pred_proba, axis=1)
y_test_pred_nn = np.argmax(y_test_pred_proba, axis=1)

# Convert back to original label format
reverse_mapping = {0: -1, 1: 0, 2: 1}
y_val_pred_orig = pd.Series(y_val_pred_nn).map(reverse_mapping)
y_test_pred_orig = pd.Series(y_test_pred_nn).map(reverse_mapping)

# Calculate metrics
val_metrics_nn = {
    'accuracy': accuracy_score(y_val, y_val_pred_orig),
    'balanced_accuracy': balanced_accuracy_score(y_val, y_val_pred_orig),
    'f1_macro': f1_score(y_val, y_val_pred_orig, average='macro')
}

test_metrics_nn = {
    'accuracy': accuracy_score(y_test, y_test_pred_orig),
    'balanced_accuracy': balanced_accuracy_score(y_test, y_test_pred_orig),
    'f1_macro': f1_score(y_test, y_test_pred_orig, average='macro')
}

print(f"\\nNeural Network Results:")
print(f"  Training time: {train_time:.2f} seconds")
print(f"  Validation Accuracy: {val_metrics_nn['accuracy']:.4f}")
print(f"  Validation F1-Macro: {val_metrics_nn['f1_macro']:.4f}")
print(f"  Test Accuracy: {test_metrics_nn['accuracy']:.4f}")
print(f"  Test F1-Macro: {test_metrics_nn['f1_macro']:.4f}")

# Add to results
results['Neural Network'] = {
    'model': model,
    'train_time': train_time,
    'val_metrics': val_metrics_nn,
    'test_metrics': test_metrics_nn,
    'val_predictions': y_val_pred_orig,
    'test_predictions': y_test_pred_orig,
    'history': history
}

# Plot training history
print("\\nPlotting training history...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Loss plot
ax1.plot(history.history['loss'], label='Training Loss', color='blue')
ax1.plot(history.history['val_loss'], label='Validation Loss', color='red')
ax1.set_title('Model Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(history.history['accuracy'], label='Training Accuracy', color='blue')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', color='red')
ax2.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Update comparison table with neural network
comparison_data = []
for model_name, result in results.items():
    comparison_data.append({
        'Model': model_name,
        'Train_Time_s': result['train_time'],
        'Val_Accuracy': result['val_metrics']['accuracy'],
        'Val_Balanced_Acc': result['val_metrics']['balanced_accuracy'],
        'Val_F1_Macro': result['val_metrics']['f1_macro'],
        'Test_Accuracy': result['test_metrics']['accuracy'],
        'Test_Balanced_Acc': result['test_metrics']['balanced_accuracy'],
        'Test_F1_Macro': result['test_metrics']['f1_macro']
    })

comparison_df_updated = pd.DataFrame(comparison_data)
comparison_df_updated = comparison_df_updated.round(4)

print("\\n" + "="*60)
print("UPDATED MODEL COMPARISON (Including Neural Network)")
print("="*60)
print(comparison_df_updated)

# Find best model
best_model_name = comparison_df_updated.loc[comparison_df_updated['Val_F1_Macro'].idxmax(), 'Model']
best_score = comparison_df_updated['Val_F1_Macro'].max()

print(f"\\n🏆 Best model so far: {best_model_name}")
print(f"   Validation F1-Macro: {best_score:.4f}")
print(f"\\n✅ Neural network trained and evaluated!")

## Cell 10: Hyperparameter Tuning (Optional but Recommended)

In [ ]:
# Cell 10: Hyperparameter Tuning (Optional but Recommended)

print("="*50)
print("HYPERPARAMETER TUNING WITH OPTUNA")
print("="*50)

# Define objective function for XGBoost tuning
def objective_xgb(trial):
    # Suggest hyperparameters
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'random_state': RANDOM_SEED,
        'eval_metric': 'mlogloss',
        'n_jobs': -1
    }
    
    # Train model
    model = xgb.XGBClassifier(**params)
    model.fit(X_train_scaled, y_train.map({-1: 0, 0: 1, 1: 2}))
    
    # Predict and evaluate
    y_pred = model.predict(X_val_scaled)
    y_pred_orig = pd.Series(y_pred).map({0: -1, 1: 0, 2: 1})
    
    # Return F1-macro score (to maximize)
    return f1_score(y_val, y_pred_orig, average='macro')

# Define objective function for MLP tuning
def objective_mlp(trial):
    # Suggest hyperparameters
    n_layers = trial.suggest_int('n_layers', 2, 4)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    
    # Build model
    model = Sequential()
    model.add(Dense(trial.suggest_int('units_1', 64, 256), activation='relu', 
                   input_shape=(X_train_scaled.shape[1],)))
    model.add(Dropout(dropout_rate))
    
    for i in range(n_layers - 1):
        model.add(Dense(trial.suggest_int(f'units_{i+2}', 32, 128), activation='relu'))
        model.add(Dropout(dropout_rate))
    
    model.add(Dense(3, activation='softmax'))
    
    # Compile
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                 loss='categorical_crossentropy', metrics=['accuracy'])
    
    # Train with early stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
    model.fit(X_train_scaled, to_categorical(y_train.map({-1: 0, 0: 1, 1: 2}), 3),
             validation_data=(X_val_scaled, to_categorical(y_val.map({-1: 0, 0: 1, 1: 2}), 3)),
             epochs=50, batch_size=32, callbacks=[early_stop], verbose=0)
    
    # Predict and evaluate
    y_pred_proba = model.predict(X_val_scaled, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_pred_orig = pd.Series(y_pred).map({0: -1, 1: 0, 2: 1})
    
    return f1_score(y_val, y_pred_orig, average='macro')

# Tune XGBoost
print("Tuning XGBoost hyperparameters...")
study_xgb = optuna.create_study(direction='maximize', study_name='xgb_tuning')
study_xgb.optimize(objective_xgb, n_trials=20, show_progress_bar=True)

print(f"Best XGBoost parameters: {study_xgb.best_params}")
print(f"Best XGBoost score: {study_xgb.best_value:.4f}")

# Tune MLP
print("\\nTuning MLP hyperparameters...")
study_mlp = optuna.create_study(direction='maximize', study_name='mlp_tuning')
study_mlp.optimize(objective_mlp, n_trials=15, show_progress_bar=True)

print(f"Best MLP parameters: {study_mlp.best_params}")
print(f"Best MLP score: {study_mlp.best_value:.4f}")

# Train best models
print("\\nTraining optimized models...")

# Best XGBoost
best_xgb = xgb.XGBClassifier(**study_xgb.best_params)
best_xgb.fit(X_train_scaled, y_train.map({-1: 0, 0: 1, 1: 2}))

# Best MLP
best_mlp_params = study_mlp.best_params.copy()
n_layers = best_mlp_params.pop('n_layers')
dropout_rate = best_mlp_params.pop('dropout_rate')
learning_rate = best_mlp_params.pop('learning_rate')

best_mlp = Sequential()
best_mlp.add(Dense(best_mlp_params['units_1'], activation='relu', input_shape=(X_train_scaled.shape[1],)))
best_mlp.add(Dropout(dropout_rate))

for i in range(n_layers - 1):
    units_key = f'units_{i+2}'
    if units_key in best_mlp_params:
        best_mlp.add(Dense(best_mlp_params[units_key], activation='relu'))
        best_mlp.add(Dropout(dropout_rate))

best_mlp.add(Dense(3, activation='softmax'))
best_mlp.compile(optimizer=Adam(learning_rate=learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
best_mlp.fit(X_train_scaled, to_categorical(y_train.map({-1: 0, 0: 1, 1: 2}), 3),
            validation_data=(X_val_scaled, to_categorical(y_val.map({-1: 0, 0: 1, 1: 2}), 3)),
            epochs=100, batch_size=32, callbacks=[early_stop], verbose=0)

print("✅ Hyperparameter tuning completed!")
print(f"XGBoost improvement: {study_xgb.best_value - results['XGBoost']['val_metrics']['f1_macro']:.4f}")
print(f"MLP improvement: {study_mlp.best_value - results['Neural Network']['val_metrics']['f1_macro']:.4f}")

## Cell 11: Model Interpretation

In [ ]:
# Cell 11: Model Interpretation

print("="*50)
print("MODEL INTERPRETATION")
print("="*50)

# Use the best performing non-DL model for interpretation
best_traditional_model = None
best_traditional_score = 0
best_traditional_name = ""

for name, result in results.items():
    if name != 'Neural Network' and result['val_metrics']['f1_macro'] > best_traditional_score:
        best_traditional_score = result['val_metrics']['f1_macro']
        best_traditional_model = result['model']
        best_traditional_name = name

print(f"Using {best_traditional_name} for interpretation (F1-Macro: {best_traditional_score:.4f})")

# 1. Feature Importance Plot
print("\\n1. Feature Importance Analysis")
if hasattr(best_traditional_model, 'feature_importances_'):
    # Tree-based model
    importances = best_traditional_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    # Plot top 15 features
    plt.figure(figsize=(12, 8))
    top_features = feature_importance_df.head(15)
    plt.barh(range(len(top_features)), top_features['importance'], color='skyblue')
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Feature Importance')
    plt.title(f'Top 15 Feature Importances - {best_traditional_name}', fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("Top 10 most important features:")
    for i, (_, row) in enumerate(feature_importance_df.head(10).iterrows(), 1):
        print(f"{i:2d}. {row['feature']:<30} {row['importance']:.4f}")

elif hasattr(best_traditional_model, 'coef_'):
    # Linear model
    importances = np.abs(best_traditional_model.coef_).mean(axis=0)
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    # Plot top 15 features
    plt.figure(figsize=(12, 8))
    top_features = feature_importance_df.head(15)
    plt.barh(range(len(top_features)), top_features['importance'], color='lightcoral')
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Coefficient Magnitude')
    plt.title(f'Top 15 Feature Coefficients - {best_traditional_name}', fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

# 2. SHAP Analysis
print("\\n2. SHAP Analysis")
try:
    # Sample data for SHAP (use subset for performance)
    sample_size = min(1000, len(X_val_scaled))
    sample_indices = np.random.choice(len(X_val_scaled), sample_size, replace=False)
    X_sample = X_val_scaled.iloc[sample_indices]
    
    if best_traditional_name == 'XGBoost':
        # For XGBoost, we need to handle the label mapping
        explainer = shap.TreeExplainer(best_traditional_model)
        shap_values = explainer.shap_values(X_sample)
        
        # SHAP summary plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False, max_display=15)
        plt.title('SHAP Summary Plot - XGBoost', fontweight='bold')
        plt.tight_layout()
        plt.show()
        
    elif best_traditional_name == 'Random Forest':
        explainer = shap.TreeExplainer(best_traditional_model)
        shap_values = explainer.shap_values(X_sample)
        
        # For multi-class, show SHAP values for each class
        plt.figure(figsize=(15, 10))
        for i, class_name in enumerate(['Down', 'Flat', 'Up']):
            plt.subplot(2, 2, i+1)
            shap.summary_plot(shap_values[i], X_sample, feature_names=feature_names, 
                            show=False, max_display=10)
            plt.title(f'SHAP - {class_name} Class', fontweight='bold')
        plt.tight_layout()
        plt.show()
        
    elif best_traditional_name == 'Logistic Regression':
        explainer = shap.LinearExplainer(best_traditional_model, X_train_scaled)
        shap_values = explainer.shap_values(X_sample)
        
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False, max_display=15)
        plt.title('SHAP Summary Plot - Logistic Regression', fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    print("✅ SHAP analysis completed successfully")
    
except Exception as e:
    print(f"SHAP analysis failed: {e}")
    print("Falling back to permutation importance...")
    
    # Permutation importance as fallback
    from sklearn.inspection import permutation_importance
    
    perm_importance = permutation_importance(
        best_traditional_model, X_val_scaled, y_val.map({-1: 0, 0: 1, 1: 2}) if best_traditional_name == 'XGBoost' else y_val,
        n_repeats=5, random_state=RANDOM_SEED, scoring='f1_macro'
    )
    
    perm_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': perm_importance.importances_mean,
        'std': perm_importance.importances_std
    }).sort_values('importance', ascending=False)
    
    # Plot permutation importance
    plt.figure(figsize=(12, 8))
    top_features = perm_importance_df.head(15)
    plt.barh(range(len(top_features)), top_features['importance'], 
             xerr=top_features['std'], color='lightgreen', alpha=0.7)
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Permutation Importance')
    plt.title(f'Permutation Importance - {best_traditional_name}', fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

# 3. Neural Network Interpretation (if applicable)
if 'Neural Network' in results:
    print("\\n3. Neural Network Interpretation")
    try:
        from sklearn.inspection import permutation_importance
        
        # Create a wrapper for the neural network to work with sklearn
        class KerasWrapper:
            def __init__(self, model):
                self.model = model
            
            def predict(self, X):
                proba = self.model.predict(X, verbose=0)
                return np.argmax(proba, axis=1)
        
        nn_wrapper = KerasWrapper(results['Neural Network']['model'])
        
        # Permutation importance for neural network
        perm_importance_nn = permutation_importance(
            nn_wrapper, X_val_scaled, y_val.map({-1: 0, 0: 1, 1: 2}),
            n_repeats=3, random_state=RANDOM_SEED, scoring='f1_macro'
        )
        
        perm_importance_nn_df = pd.DataFrame({
            'feature': feature_names,
            'importance': perm_importance_nn.importances_mean,
            'std': perm_importance_nn.importances_std
        }).sort_values('importance', ascending=False)
        
        # Plot neural network permutation importance
        plt.figure(figsize=(12, 8))
        top_features_nn = perm_importance_nn_df.head(15)
        plt.barh(range(len(top_features_nn)), top_features_nn['importance'], 
                 xerr=top_features_nn['std'], color='purple', alpha=0.7)
        plt.yticks(range(len(top_features_nn)), top_features_nn['feature'])
        plt.xlabel('Permutation Importance')
        plt.title('Neural Network - Permutation Importance', fontweight='bold')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
        
        print("✅ Neural network interpretation completed")
        
    except Exception as e:
        print(f"Neural network interpretation failed: {e}")

print("\\n" + "="*50)
print("INTERPRETATION INSIGHTS")
print("="*50)
print(f"• Best traditional model: {best_traditional_name}")
print(f"• Most important features reveal key price movement predictors")
print(f"• SHAP values show feature contribution directions")
print(f"• Technical indicators vs. raw features importance balance")
print("• Review plots above for detailed feature behavior analysis")

## Cell 12: Final Evaluation & Strategy Simulation

In [ ]:
# Cell 12: Final Evaluation & Strategy Simulation

print("="*50)
print("FINAL EVALUATION & STRATEGY SIMULATION")
print("="*50)

# Find the best overall model
best_overall_model = None
best_overall_score = 0
best_overall_name = ""

for name, result in results.items():
    if result['val_metrics']['f1_macro'] > best_overall_score:
        best_overall_score = result['val_metrics']['f1_macro']
        best_overall_model = result['model']
        best_overall_name = name

print(f"Best overall model: {best_overall_name} (Val F1-Macro: {best_overall_score:.4f})")

# Final test set evaluation
print(f"\\nFinal Test Set Evaluation:")
print("-" * 40)

if best_overall_name == 'XGBoost':
    y_test_pred = best_overall_model.predict(X_test_scaled)
    y_test_pred_final = pd.Series(y_test_pred).map({0: -1, 1: 0, 2: 1})
elif best_overall_name == 'Neural Network':
    y_test_pred_proba = best_overall_model.predict(X_test_scaled, verbose=0)
    y_test_pred = np.argmax(y_test_pred_proba, axis=1)
    y_test_pred_final = pd.Series(y_test_pred).map({0: -1, 1: 0, 2: 1})
else:
    y_test_pred_final = best_overall_model.predict(X_test_scaled)

# Calculate final metrics
final_accuracy = accuracy_score(y_test, y_test_pred_final)
final_balanced_acc = balanced_accuracy_score(y_test, y_test_pred_final)
final_f1_macro = f1_score(y_test, y_test_pred_final, average='macro')

print(f"Test Accuracy: {final_accuracy:.4f}")
print(f"Test Balanced Accuracy: {final_balanced_acc:.4f}")
print(f"Test F1-Macro: {final_f1_macro:.4f}")

# Detailed classification report
print(f"\\nDetailed Classification Report:")
print(classification_report(y_test, y_test_pred_final, target_names=['Down', 'Flat', 'Up']))

# Confusion Matrix
print(f"\\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_test_pred_final)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Down', 'Flat', 'Up'], 
            yticklabels=['Down', 'Flat', 'Up'])
plt.title(f'Confusion Matrix - {best_overall_name}', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

# Strategy Simulation
print(f"\\n" + "="*30)
print("TRADING STRATEGY SIMULATION")
print("="*30)

# Get test set data with predictions
test_data = df.iloc[len(X_train) + len(X_val):].copy()
test_data['prediction'] = y_test_pred_final.values
test_data['actual'] = y_test.values

# Simple trading strategy
def simulate_strategy(data, initial_capital=10000):
    """
    Simulate a simple trading strategy:
    - Long when prediction = 1 (Up)
    - Short when prediction = -1 (Down)  
    - Hold cash when prediction = 0 (Flat)
    """
    capital = initial_capital
    position = 0  # 0: cash, 1: long, -1: short
    portfolio_values = [capital]
    trades = []
    
    for i in range(len(data)):
        current_price = data['close'].iloc[i]
        pred = data['prediction'].iloc[i]
        
        if i == 0:
            prev_price = current_price
        else:
            prev_price = data['close'].iloc[i-1]
        
        # Calculate return from previous position
        if position == 1:  # Long position
            capital *= (current_price / prev_price)
        elif position == -1:  # Short position
            capital *= (prev_price / current_price)
        
        # Make new position decision
        new_position = pred
        
        if new_position != position:
            trades.append({
                'date': data.index[i],
                'action': 'Long' if new_position == 1 else 'Short' if new_position == -1 else 'Cash',
                'price': current_price,
                'capital': capital
            })
            position = new_position
        
        portfolio_values.append(capital)
    
    return portfolio_values[:-1], trades

# Run strategy simulation
portfolio_values, trades = simulate_strategy(test_data)

# Calculate buy-and-hold benchmark
initial_price = test_data['close'].iloc[0]
final_price = test_data['close'].iloc[-1]
buy_hold_return = (final_price / initial_price - 1) * 100

# Calculate strategy return
strategy_return = (portfolio_values[-1] / 10000 - 1) * 100

print(f"Strategy Performance:")
print(f"  Initial Capital: $10,000")
print(f"  Final Portfolio Value: ${portfolio_values[-1]:,.2f}")
print(f"  Strategy Return: {strategy_return:.2f}%")
print(f"  Buy-and-Hold Return: {buy_hold_return:.2f}%")
print(f"  Excess Return: {strategy_return - buy_hold_return:.2f}%")
print(f"  Number of Trades: {len(trades)}")

# Plot equity curves
plt.figure(figsize=(15, 8))

# Strategy equity curve
dates = test_data.index
plt.plot(dates, portfolio_values, label=f'ML Strategy ({strategy_return:.1f}%)', 
         linewidth=2, color='blue')

# Buy-and-hold benchmark
buy_hold_values = [10000 * (test_data['close'].iloc[i] / initial_price) 
                   for i in range(len(test_data))]
plt.plot(dates, buy_hold_values, label=f'Buy & Hold ({buy_hold_return:.1f}%)', 
         linewidth=2, color='red', alpha=0.7)

plt.title('Trading Strategy Performance vs Buy-and-Hold', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Portfolio Value ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Show sample trades
print(f"\\nSample Trades (first 10):")
trades_df = pd.DataFrame(trades)
if len(trades_df) > 0:
    print(trades_df.head(10))
else:
    print("No trades executed")

# Strategy statistics
if len(portfolio_values) > 1:
    returns = pd.Series(portfolio_values).pct_change().dropna()
    volatility = returns.std() * np.sqrt(252) * 100  # Annualized
    sharpe_ratio = (strategy_return / 100) / (volatility / 100) if volatility > 0 else 0
    max_drawdown = ((pd.Series(portfolio_values).cummax() - pd.Series(portfolio_values)) / pd.Series(portfolio_values).cummax()).max() * 100
    
    print(f"\\nStrategy Statistics:")
    print(f"  Annualized Volatility: {volatility:.2f}%")
    print(f"  Sharpe Ratio: {sharpe_ratio:.2f}")
    print(f"  Maximum Drawdown: {max_drawdown:.2f}%")

print(f"\\n✅ Final evaluation and strategy simulation completed!")

## Cell: Model Saving System

In [ ]:
# Cell: Model Saving System

import os
import json
import pickle
from datetime import datetime
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend for saving plots

print("="*50)
print("MODEL SAVING SYSTEM")
print("="*50)

# Create timestamped folder
timestamp = datetime.now().strftime("%Y_%m_%d__%H_%M_%S")
save_dir = f"models/{timestamp}"
os.makedirs(save_dir, exist_ok=True)

# Create subdirectories
subdirs = ['models', 'plots', 'stats', 'data', 'config']
for subdir in subdirs:
    os.makedirs(os.path.join(save_dir, subdir), exist_ok=True)

print(f"Created save directory: {save_dir}")

# 1. Save Configuration and Metadata
print("\n1. Saving configuration and metadata...")
config = {
    'timestamp': timestamp,
    'experiment_config': {
        'N': N,
        'P_PCT': P_PCT,
        'RANDOM_SEED': RANDOM_SEED,
        'TEST_SIZE': TEST_SIZE,
        'VAL_SIZE': VAL_SIZE
    },
    'data_info': {
        'total_samples': len(df),
        'n_features': len(feature_cols),
        'feature_names': feature_cols,
        'train_size': len(X_train),
        'val_size': len(X_val),
        'test_size': len(X_test),
        'class_distribution': df['label'].value_counts().to_dict()
    },
    'model_results': {}
}

# Add model results to config
for model_name, result in results.items():
    config['model_results'][model_name] = {
        'train_time_seconds': result['train_time'],
        'val_metrics': result['val_metrics'],
        'test_metrics': result['test_metrics']
    }

# Save configuration
with open(os.path.join(save_dir, 'config', 'experiment_config.json'), 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, default=str)

print(f"   ✅ Saved experiment configuration")

# 2. Save All Trained Models
print("\n2. Saving trained models...")
saved_models = {}

for model_name, result in results.items():
    model = result['model']
    model_filename = f"{model_name.lower().replace(' ', '_')}.pkl"
    model_path = os.path.join(save_dir, 'models', model_filename)
    
    try:
        if model_name == 'Neural Network':
            # Save Keras model
            keras_path = os.path.join(save_dir, 'models', 'neural_network.h5')
            model.save(keras_path)
            saved_models[model_name] = keras_path
            print(f"   ✅ Saved {model_name} (Keras format)")
        else:
            # Save sklearn/xgboost models with joblib
            joblib.dump(model, model_path)
            saved_models[model_name] = model_path
            print(f"   ✅ Saved {model_name}")
    except Exception as e:
        print(f"   ❌ Failed to save {model_name}: {e}")

# Save scaler
scaler_path = os.path.join(save_dir, 'models', 'scaler.pkl')
joblib.dump(scaler, scaler_path)
saved_models['Scaler'] = scaler_path
print(f"   ✅ Saved StandardScaler")

# 3. Save Model Statistics and Comparison Tables
print("\n3. Saving model statistics...")

# Model comparison table
comparison_df_updated.to_csv(os.path.join(save_dir, 'stats', 'model_comparison.csv'), index=False)
print(f"   ✅ Saved model comparison table")

# Feature importance (for tree-based models)
if 'Random Forest' in results:
    rf_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': results['Random Forest']['model'].feature_importances_
    }).sort_values('importance', ascending=False)
    rf_importance.to_csv(os.path.join(save_dir, 'stats', 'random_forest_feature_importance.csv'), index=False)
    print(f"   ✅ Saved Random Forest feature importance")

if 'XGBoost' in results:
    xgb_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': results['XGBoost']['model'].feature_importances_
    }).sort_values('importance', ascending=False)
    xgb_importance.to_csv(os.path.join(save_dir, 'stats', 'xgboost_feature_importance.csv'), index=False)
    print(f"   ✅ Saved XGBoost feature importance")

# Mutual information scores
feature_importance.to_csv(os.path.join(save_dir, 'stats', 'mutual_information_scores.csv'), index=False)
print(f"   ✅ Saved mutual information scores")

# Class distribution statistics
class_stats = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test', 'Overall'],
    'Down_Count': [
        y_train.value_counts().get(-1, 0),
        y_val.value_counts().get(-1, 0), 
        y_test.value_counts().get(-1, 0),
        df['label'].value_counts().get(-1, 0)
    ],
    'Flat_Count': [
        y_train.value_counts().get(0, 0),
        y_val.value_counts().get(0, 0),
        y_test.value_counts().get(0, 0),
        df['label'].value_counts().get(0, 0)
    ],
    'Up_Count': [
        y_train.value_counts().get(1, 0),
        y_val.value_counts().get(1, 0),
        y_test.value_counts().get(1, 0),
        df['label'].value_counts().get(1, 0)
    ]
})
class_stats.to_csv(os.path.join(save_dir, 'stats', 'class_distribution.csv'), index=False)
print(f"   ✅ Saved class distribution statistics")

# 4. Save All Plots and Visualizations
print("\n4. Saving plots and visualizations...")

# Function to save current figure
def save_plot(filename, dpi=300):
    plt.savefig(os.path.join(save_dir, 'plots', filename), 
                dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close()

# Recreate and save key plots
plt.style.use('seaborn-v0_8-whitegrid')

# 4.1 Model Comparison Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics = ['Val_Accuracy', 'Val_Balanced_Acc', 'Val_F1_Macro']
metric_names = ['Accuracy', 'Balanced Accuracy', 'F1-Macro']

for i, (metric, name) in enumerate(zip(metrics, metric_names)):
    ax = axes[i]
    bars = ax.bar(comparison_df_updated['Model'], comparison_df_updated[metric], 
                  color=['lightblue', 'lightgreen', 'lightcoral', 'lightyellow'], alpha=0.7)
    ax.set_title(f'Validation {name}', fontsize=12, fontweight='bold')
    ax.set_ylabel(name, fontsize=10)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    
    for bar, value in zip(bars, comparison_df_updated[metric]):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
save_plot('model_comparison.png')
print(f"   ✅ Saved model comparison plot")

# 4.2 Feature Importance Plot (Top 20)
fig, ax = plt.subplots(figsize=(12, 8))
top_20_features = feature_importance.head(20)
bars = ax.barh(range(len(top_20_features)), top_20_features['mi_score'])
ax.set_yticks(range(len(top_20_features)))
ax.set_yticklabels(top_20_features['feature'])
ax.set_xlabel('Mutual Information Score')
ax.set_title('Top 20 Most Predictive Features', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_plot('feature_importance_top20.png')
print(f"   ✅ Saved feature importance plot")

# 4.3 Class Distribution Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Bar plot
labels = ['Down', 'Flat', 'Up']
counts = [df['label'].value_counts().get(i, 0) for i in [-1, 0, 1]]
colors = ['red', 'gray', 'green']

bars = ax1.bar(labels, counts, color=colors, alpha=0.7)
ax1.set_title('Overall Class Distribution', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Samples', fontsize=12)
ax1.grid(True, alpha=0.3)

for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
             f'{count}', ha='center', va='bottom', fontweight='bold')

# Pie chart
props = [count/sum(counts) for count in counts]
ax2.pie(props, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax2.set_title('Class Distribution (Proportions)', fontsize=14, fontweight='bold')

plt.tight_layout()
save_plot('class_distribution.png')
print(f"   ✅ Saved class distribution plot")

# 4.4 Training History (if Neural Network exists)
if 'Neural Network' in results and 'history' in results['Neural Network']:
    history = results['Neural Network']['history']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Loss plot
    ax1.plot(history.history['loss'], label='Training Loss', color='blue')
    ax1.plot(history.history['val_loss'], label='Validation Loss', color='red')
    ax1.set_title('Neural Network Training Loss', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy plot
    ax2.plot(history.history['accuracy'], label='Training Accuracy', color='blue')
    ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', color='red')
    ax2.set_title('Neural Network Training Accuracy', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    save_plot('neural_network_training_history.png')
    print(f"   ✅ Saved neural network training history")

# 4.5 Data Split Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Split sizes
splits = ['Train', 'Validation', 'Test']
sizes = [len(y_train), len(y_val), len(y_test)]
colors_split = ['lightblue', 'lightgreen', 'lightcoral']

ax1.bar(splits, sizes, color=colors_split, alpha=0.7)
ax1.set_title('Dataset Split Sizes', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Samples', fontsize=12)
ax1.grid(True, alpha=0.3)

for i, (split, size) in enumerate(zip(splits, sizes)):
    ax1.text(i, size + size*0.01, f'{size}\\n({size/len(df):.1%})', 
             ha='center', va='bottom', fontweight='bold')

# Class distribution by split
x = np.arange(len(splits))
width = 0.25

down_counts = [y_train.value_counts().get(-1, 0), y_val.value_counts().get(-1, 0), y_test.value_counts().get(-1, 0)]
flat_counts = [y_train.value_counts().get(0, 0), y_val.value_counts().get(0, 0), y_test.value_counts().get(0, 0)]
up_counts = [y_train.value_counts().get(1, 0), y_val.value_counts().get(1, 0), y_test.value_counts().get(1, 0)]

ax2.bar(x - width, down_counts, width, label='Down', color='red', alpha=0.7)
ax2.bar(x, flat_counts, width, label='Flat', color='gray', alpha=0.7)
ax2.bar(x + width, up_counts, width, label='Up', color='green', alpha=0.7)

ax2.set_title('Class Distribution by Split', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Samples', fontsize=12)
ax2.set_xlabel('Split', fontsize=12)
ax2.set_xticks(x)
ax2.set_xticklabels(splits)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_plot('data_splits.png')
print(f"   ✅ Saved data split visualization")

# 5. Save Sample Data
print("\n5. Saving sample data...")

# Save feature matrix samples
X_train.head(100).to_csv(os.path.join(save_dir, 'data', 'X_train_sample.csv'))
y_train.head(100).to_csv(os.path.join(save_dir, 'data', 'y_train_sample.csv'))
X_train.to_csv(os.path.join(save_dir, 'data', 'X_train.csv'))
y_train.to_csv(os.path.join(save_dir, 'data', 'y_train.csv'))
X_val.to_csv(os.path.join(save_dir, 'data', 'X_val.csv'))
y_val.to_csv(os.path.join(save_dir, 'data', 'y_val.csv'))
X_test.to_csv(os.path.join(save_dir, 'data', 'X_test.csv'))
y_test.to_csv(os.path.join(save_dir, 'data', 'y_test.csv'))
print(f"   ✅ Saved training data samples")

# Save predictions with proper error handling
print("\n   Preparing predictions DataFrame...")
predictions_data = {'actual': y_test}

# Check and align prediction lengths
print(f"   y_test length: {len(y_test)}")
for model_name, result in results.items():
    pred = result['test_predictions']
    print(f"   {model_name} predictions length: {len(pred)}")
    
    # Handle different prediction formats
    if hasattr(pred, 'values'):
        pred_values = pred.values
    elif isinstance(pred, (pd.Series, pd.DataFrame)):
        pred_values = pred.values.flatten() if hasattr(pred, 'values') else pred
    else:
        pred_values = np.array(pred).flatten()
    
    # Ensure predictions match y_test length
    if len(pred_values) == len(y_test):
        predictions_data[f'{model_name}_pred'] = pred_values
        print(f"   ✅ {model_name}: {len(pred_values)} predictions aligned")
    else:
        print(f"   ⚠️  {model_name}: Length mismatch ({len(pred_values)} vs {len(y_test)})")
        # Try to align by index if possible
        if hasattr(pred, 'index') and hasattr(y_test, 'index'):
            try:
                aligned_pred = pred.reindex(y_test.index, fill_value=np.nan)
                predictions_data[f'{model_name}_pred'] = aligned_pred.values
                print(f"   ✅ {model_name}: Aligned by index")
            except:
                print(f"   ❌ {model_name}: Could not align, skipping")
        else:
            print(f"   ❌ {model_name}: Could not align, skipping")

try:
    predictions_df = pd.DataFrame(predictions_data)
    predictions_df.to_csv(os.path.join(save_dir, 'data', 'test_predictions.csv'), index=False)
    print(f"   ✅ Saved test predictions ({len(predictions_df)} rows, {len(predictions_df.columns)} columns)")
except Exception as e:
    print(f"   ❌ Failed to save predictions: {e}")
    # Save individual prediction files as backup
    for model_name, result in results.items():
        try:
            pred_df = pd.DataFrame({
                'prediction': result['test_predictions']
            })
            pred_filename = f"{model_name.lower().replace(' ', '_')}_predictions.csv"
            pred_df.to_csv(os.path.join(save_dir, 'data', pred_filename), index=False)
            print(f"   ✅ Saved {model_name} predictions separately")
        except Exception as e2:
            print(f"   ❌ Failed to save {model_name} predictions: {e2}")

# 6. Create Summary Report
print("\n6. Creating summary report...")
best_model_name = comparison_df_updated.loc[comparison_df_updated['Val_F1_Macro'].idxmax(), 'Model']
best_score = comparison_df_updated['Val_F1_Macro'].max()

summary_report = f"""# OHLCV Time Series Classification - Experiment Report

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Experiment ID: {timestamp}

## Configuration
- Prediction Horizon (N): {N} bars
- Price Threshold: {P_PCT}%
- Random Seed: {RANDOM_SEED}
- Test Size: {TEST_SIZE*100}%
- Validation Size: {VAL_SIZE*100}%

## Dataset Summary
- Total Samples: {len(df):,}
- Features: {len(feature_cols)}
- Training Samples: {len(X_train):,}
- Validation Samples: {len(X_val):,}
- Test Samples: {len(X_test):,}

## Class Distribution
- Down (-1): {df['label'].value_counts().get(-1, 0):,} ({df['label'].value_counts(normalize=True).get(-1, 0):.1%})
- Flat (0): {df['label'].value_counts().get(0, 0):,} ({df['label'].value_counts(normalize=True).get(0, 0):.1%})
- Up (1): {df['label'].value_counts().get(1, 0):,} ({df['label'].value_counts(normalize=True).get(1, 0):.1%})

## Model Results (Validation F1-Macro)
"""

for _, row in comparison_df_updated.iterrows():
    summary_report += f"- {row['Model']}: {row['Val_F1_Macro']:.4f}\\n"

summary_report += f"""
## Best Model
WINNER: {best_model_name} (F1-Macro: {best_score:.4f})

## Files Saved
- Models: {len(saved_models)} files in models/
- Plots: {len([f for f in os.listdir(os.path.join(save_dir, 'plots')) if f.endswith('.png')])} PNG files in plots/
- Statistics: {len([f for f in os.listdir(os.path.join(save_dir, 'stats')) if f.endswith('.csv')])} CSV files in stats/
- Configuration: experiment_config.json in config/
- Sample Data: Training samples and predictions in data/

## Directory Structure
```
{save_dir}/
├── models/          # Trained models (.pkl, .h5)
├── plots/           # Visualizations (.png)
├── stats/           # Statistics and metrics (.csv)
├── data/            # Sample data and predictions
├── config/          # Experiment configuration
└── README.md        # This summary report
```

## Model Files
"""

for model_name, path in saved_models.items():
    summary_report += f"- {model_name}: {os.path.basename(path)}\\n"

summary_report += f"""
## Usage Instructions

### Loading Models
```python
import joblib
from tensorflow.keras.models import load_model

# Load scaler
scaler = joblib.load('{os.path.join(save_dir, 'models', 'scaler.pkl')}')

# Load best model ({best_model_name})
"""

if best_model_name == 'Neural Network':
    summary_report += f"model = load_model('{os.path.join(save_dir, 'models', 'neural_network.h5')}')"
else:
    model_file = f"{best_model_name.lower().replace(' ', '_')}.pkl"
    summary_report += f"model = joblib.load('{os.path.join(save_dir, 'models', model_file)}')"

summary_report += f"""
```

### Making Predictions
```python
# Scale new data
X_new_scaled = scaler.transform(X_new)

# Make predictions
predictions = model.predict(X_new_scaled)
```

## Experiment Notes
- All models trained with balanced class weights
- Temporal data splitting used (no data leakage)
- Feature scaling applied to training data only
- {len(feature_cols)} engineered features used
- Best validation F1-macro score: {best_score:.4f}

Generated by OHLCV Time Series Classification Pipeline
"""

# Save summary report with UTF-8 encoding
with open(os.path.join(save_dir, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(summary_report)

print(f"   ✅ Saved summary report")

# 7. Final Summary
print("\n" + "="*50)
print("SAVING COMPLETE!")
print("="*50)
print(f"[FOLDER] All files saved to: {save_dir}")
print(f"[WINNER] Best model: {best_model_name} (F1-Macro: {best_score:.4f})")
print(f"[MODELS] {len(saved_models)} models saved")
print(f"[PLOTS] {len([f for f in os.listdir(os.path.join(save_dir, 'plots')) if f.endswith('.png')])} plots saved")
print(f"[STATS] {len([f for f in os.listdir(os.path.join(save_dir, 'stats')) if f.endswith('.csv')])} statistics files saved")

# Create a quick access dictionary for loading models later
model_paths = {
    'save_directory': save_dir,
    'timestamp': timestamp,
    'best_model': best_model_name,
    'model_files': saved_models,
    'config_file': os.path.join(save_dir, 'config', 'experiment_config.json'),
    'summary_report': os.path.join(save_dir, 'README.md')
}

print(f"\n[INFO] Quick access info:")
for key, value in model_paths.items():
    print(f"   {key}: {value}")

print(f"\n[SUCCESS] Complete experiment saved successfully!")
print(f"   Use the paths above to load models and results later.")

## Cell: Model Loading Utilities

In [ ]:
# Cell: Model Loading Utilities

print("="*50)
print("MODEL LOADING UTILITIES")
print("="*50)

def load_experiment(experiment_path):
    """
    Load a complete experiment from a saved directory.
    
    Args:
        experiment_path (str): Path to the experiment directory
        
    Returns:
        dict: Dictionary containing loaded models, config, and data
    """
    import os
    import json
    import joblib
    import pandas as pd
    from tensorflow.keras.models import load_model
    
    if not os.path.exists(experiment_path):
        raise FileNotFoundError(f"Experiment directory not found: {experiment_path}")
    
    print(f"Loading experiment from: {experiment_path}")
    
    # Load configuration
    config_path = os.path.join(experiment_path, 'config', 'experiment_config.json')
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    # Load models
    models = {}
    models_dir = os.path.join(experiment_path, 'models')
    
    # Load sklearn/xgboost models
    for model_file in os.listdir(models_dir):
        if model_file.endswith('.pkl'):
            model_name = model_file.replace('.pkl', '').replace('_', ' ').title()
            if model_name == 'Scaler':
                models['scaler'] = joblib.load(os.path.join(models_dir, model_file))
            else:
                models[model_name] = joblib.load(os.path.join(models_dir, model_file))
            print(f"   ✅ Loaded {model_name}")
    
    # Load Keras model if exists
    keras_path = os.path.join(models_dir, 'neural_network.h5')
    if os.path.exists(keras_path):
        models['Neural Network'] = load_model(keras_path)
        print(f"   ✅ Loaded Neural Network")
    
    # Load statistics
    stats = {}
    stats_dir = os.path.join(experiment_path, 'stats')
    for stat_file in os.listdir(stats_dir):
        if stat_file.endswith('.csv'):
            stat_name = stat_file.replace('.csv', '')
            stats[stat_name] = pd.read_csv(os.path.join(stats_dir, stat_file))
    
    # Load sample data
    data = {}
    data_dir = os.path.join(experiment_path, 'data')
    if os.path.exists(data_dir):
        for data_file in os.listdir(data_dir):
            if data_file.endswith('.csv'):
                data_name = data_file.replace('.csv', '')
                data[data_name] = pd.read_csv(os.path.join(data_dir, data_file))
    
    return {
        'config': config,
        'models': models,
        'statistics': stats,
        'data': data,
        'experiment_path': experiment_path
    }

def list_experiments(base_dir='models'):
    """
    List all available experiments in the models directory.
    
    Args:
        base_dir (str): Base directory containing experiment folders
        
    Returns:
        list: List of experiment directories with timestamps
    """
    if not os.path.exists(base_dir):
        print(f"Models directory not found: {base_dir}")
        return []
    
    experiments = []
    for item in os.listdir(base_dir):
        item_path = os.path.join(base_dir, item)
        if os.path.isdir(item_path) and '_' in item:  # Check for timestamp format
            # Try to parse timestamp
            try:
                timestamp_parts = item.split('__')
                if len(timestamp_parts) == 2:
                    date_part, time_part = timestamp_parts
                    datetime.strptime(f"{date_part}_{time_part}", "%Y_%m_%d_%H_%M_%S")
                    experiments.append(item)
            except ValueError:
                continue
    
    experiments.sort(reverse=True)  # Most recent first
    return experiments

def predict_with_saved_model(model_path, scaler_path, X_new):
    """
    Make predictions using a saved model and scaler.
    
    Args:
        model_path (str): Path to the saved model
        scaler_path (str): Path to the saved scaler
        X_new (pd.DataFrame): New data to predict
        
    Returns:
        np.array: Predictions
    """
    # Load scaler and model
    scaler = joblib.load(scaler_path)
    
    if model_path.endswith('.h5'):
        from tensorflow.keras.models import load_model
        model = load_model(model_path)
        # Scale data
        X_scaled = scaler.transform(X_new)
        # Predict probabilities and convert to classes
        pred_proba = model.predict(X_scaled, verbose=0)
        predictions = np.argmax(pred_proba, axis=1)
        # Convert back to original labels
        label_map = {0: -1, 1: 0, 2: 1}
        predictions = pd.Series(predictions).map(label_map).values
    else:
        model = joblib.load(model_path)
        X_scaled = scaler.transform(X_new)
        predictions = model.predict(X_scaled)
    
    return predictions

# Example usage functions
print("\\nUtility functions defined:")
print("1. load_experiment(experiment_path) - Load complete experiment")
print("2. list_experiments(base_dir='models') - List available experiments") 
print("3. predict_with_saved_model(model_path, scaler_path, X_new) - Make predictions")

# Show available experiments
print("\\nAvailable experiments:")
experiments = list_experiments()
if experiments:
    for i, exp in enumerate(experiments, 1):
        exp_path = os.path.join('models', exp)
        config_path = os.path.join(exp_path, 'config', 'experiment_config.json')
        if os.path.exists(config_path):
            with open(config_path, 'r') as f:
                config = json.load(f)
            best_model = max(config['model_results'].items(), 
                           key=lambda x: x[1]['val_metrics']['f1_macro'])
            print(f"   {i}. {exp} - Best: {best_model[0]} (F1: {best_model[1]['val_metrics']['f1_macro']:.4f})")
        else:
            print(f"   {i}. {exp}")
else:
    print("   No experiments found yet. Run the model saving cell first!")

print(f"\\n✅ Model loading utilities ready!")
print(f"   Use load_experiment('{save_dir}') to reload this experiment")

## Cell 13: Conclusion & Next Steps

In [ ]:
# Cell 13: Conclusion & Next Steps

print("="*60)
print("CONCLUSION & NEXT STEPS")
print("="*60)

# Summary of findings
print("\\n📊 EXPERIMENT SUMMARY")
print("-" * 40)

# Model performance summary
print("\\nModel Performance Ranking (by Validation F1-Macro):")
performance_summary = []
for name, result in results.items():
    performance_summary.append({
        'Model': name,
        'Val_F1_Macro': result['val_metrics']['f1_macro'],
        'Test_F1_Macro': result['test_metrics']['f1_macro'],
        'Train_Time': result['train_time']
    })

performance_df = pd.DataFrame(performance_summary).sort_values('Val_F1_Macro', ascending=False)
for i, (_, row) in enumerate(performance_df.iterrows(), 1):
    print(f"{i}. {row['Model']:<20} Val: {row['Val_F1_Macro']:.4f} | Test: {row['Test_F1_Macro']:.4f} | Time: {row['Train_Time']:.1f}s")

# Best model analysis
best_model = performance_df.iloc[0]
print(f"\\n🏆 BEST MODEL: {best_model['Model']}")
print(f"   Validation F1-Macro: {best_model['Val_F1_Macro']:.4f}")
print(f"   Test F1-Macro: {best_model['Test_F1_Macro']:.4f}")
print(f"   Training Time: {best_model['Train_Time']:.1f} seconds")

# Feature importance insights
if 'feature_importance_df' in locals():
    top_3_features = feature_importance_df.head(3)['feature'].tolist()
    print(f"\\n🔍 TOP PREDICTIVE FEATURES:")
    for i, feature in enumerate(top_3_features, 1):
        print(f"   {i}. {feature}")

# Class prediction analysis
class_performance = pd.DataFrame(classification_report(y_test, y_test_pred_final, 
                                                     target_names=['Down', 'Flat', 'Up'], 
                                                     output_dict=True)).T
print(f"\\n📈 CLASS-WISE PERFORMANCE:")
for class_name in ['Down', 'Flat', 'Up']:
    if class_name in class_performance.index:
        precision = class_performance.loc[class_name, 'precision']
        recall = class_performance.loc[class_name, 'recall']
        f1 = class_performance.loc[class_name, 'f1-score']
        print(f"   {class_name:<4}: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")

# Economic significance
if 'strategy_return' in locals() and 'buy_hold_return' in locals():
    print(f"\\n💰 ECONOMIC SIGNIFICANCE:")
    print(f"   Strategy Return: {strategy_return:.2f}%")
    print(f"   Buy-and-Hold Return: {buy_hold_return:.2f}%")
    print(f"   Excess Return: {strategy_return - buy_hold_return:.2f}%")
    
    if strategy_return > buy_hold_return:
        print("   ✅ Strategy outperformed buy-and-hold")
    else:
        print("   ❌ Strategy underperformed buy-and-hold")

# Data and methodology insights
print(f"\\n📋 METHODOLOGY INSIGHTS:")
print(f"   • Dataset: {len(df)} samples with {len(feature_cols)} engineered features")
print(f"   • Prediction horizon: {N} periods ({P_PCT}% threshold)")
print(f"   • Class distribution: Down={y_test.value_counts().get(-1, 0)}, Flat={y_test.value_counts().get(0, 0)}, Up={y_test.value_counts().get(1, 0)}")
print(f"   • Temporal split maintained chronological order")
print(f"   • Feature engineering included technical indicators, lags, and rolling statistics")

print(f"\\n\\n🚀 LIMITATIONS & FUTURE WORK")
print("-" * 40)

limitations = [
    "Synthetic data may not capture real market complexities",
    "Fixed prediction horizon - adaptive horizons could improve performance", 
    "Simple trading strategy - transaction costs and slippage not considered",
    "Class imbalance may affect model performance on minority classes",
    "Feature selection could be optimized using advanced techniques",
    "Regime detection not implemented - market conditions change over time"
]

print("\\n⚠️  LIMITATIONS:")
for i, limitation in enumerate(limitations, 1):
    print(f"   {i}. {limitation}")

future_work = [
    "Implement walk-forward validation for more realistic backtesting",
    "Add regime detection and adaptive model selection",
    "Experiment with ensemble methods combining multiple models",
    "Include alternative data sources (sentiment, news, etc.)",
    "Optimize prediction horizons dynamically based on market volatility",
    "Implement more sophisticated trading strategies with risk management",
    "Add transaction cost modeling and realistic execution constraints",
    "Explore deep learning architectures (LSTM, CNN, Transformers)",
    "Implement online learning for model adaptation",
    "Add multi-asset and cross-asset feature engineering"
]

print("\\n🔮 FUTURE IMPROVEMENTS:")
for i, improvement in enumerate(future_work, 1):
    print(f"   {i:2d}. {improvement}")

print(f"\\n\\n📝 RESEARCH CONCLUSIONS")
print("-" * 40)

conclusions = [
    f"Machine learning can identify patterns in OHLCV data for price movement prediction",
    f"Technical indicators combined with statistical features provide predictive power",
    f"{best_model['Model']} achieved the best performance with {best_model['Val_F1_Macro']:.1%} F1-macro score",
    f"Feature engineering is crucial - raw price data alone insufficient",
    f"Temporal validation essential - random splits would overestimate performance",
    f"Economic significance depends on transaction costs and market impact",
    f"Class imbalance remains a challenge for minority movement classes"
]

for i, conclusion in enumerate(conclusions, 1):
    print(f"   {i}. {conclusion}")

print(f"\\n\\n✅ EXPERIMENT COMPLETED SUCCESSFULLY!")
print("="*60)
print("This notebook provides a comprehensive framework for OHLCV time series")
print("classification that can be adapted for real financial data and extended")
print("with additional features, models, and evaluation methodologies.")
print("="*60)

# Save results summary
results_summary = {
    'best_model': best_model['Model'],
    'best_val_f1': best_model['Val_F1_Macro'],
    'best_test_f1': best_model['Test_F1_Macro'],
    'dataset_size': len(df),
    'num_features': len(feature_cols),
    'prediction_horizon': N,
    'threshold_pct': P_PCT,
    'strategy_return': strategy_return if 'strategy_return' in locals() else None,
    'buy_hold_return': buy_hold_return if 'buy_hold_return' in locals() else None
}

print(f"\\n💾 Results summary saved to memory for future reference")
print(f"   Best model: {results_summary['best_model']}")
print(f"   Performance: {results_summary['best_val_f1']:.4f} (validation F1-macro)")